# 09. Paoay: prepare Dynamic World inputs

**One task:** obtain broad land-cover probability inputs for notebook 08’s exact four-month period and 10 m grid. Existing verified files are reused; only missing files can be downloaded.

No land-cover decisions, eligibility masks, crop predictions or accuracy evaluation happen here. Dynamic World is a prediction source, not ground truth. Its cloud masks and available observations differ from Sentinel-2 SR. See the [workflow guide](../../docs/paoay_workflow.md) and [official dataset](https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_DYNAMICWORLD_V1).


## 1. Setup

Finish notebook 08 first. Default `dynamic_world_mode = "download_missing"` obtains missing inputs and reuses verified files. Choose `"reuse"` for local-only inspection. Packages and settings are unchanged; no export jobs or cloud assets are created.


In [ ]:
from pathlib import Path
import json
import sys

repository_root = next((folder for folder in [Path.cwd(), *Path.cwd().parents]
                        if (folder / "pyproject.toml").is_file() and (folder / "src/crop_pipeline").is_dir()), None)
if repository_root is None:
    raise FileNotFoundError("Open this notebook from inside the project repository")
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))
from crop_pipeline.paoay import dynamic_world
from crop_pipeline.analysis_workflow import resolve_stage_pointer, write_stage_pointer
from crop_pipeline.notebook_display import show_table
from crop_pipeline import progress as pipeline_progress

settings_path = repository_root / "settings.local.json"
local_settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
def local_path(value):
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (repository_root / path).resolve()

dynamic_world_mode = "download_missing"  # Download only missing files; use "reuse" for local-only inspection.
report_folder = local_path(local_settings.get("report_root", "./sen4agrinet_work/inventory"))
paoay_root = report_folder.parent / "paoay"
imagery_pointer = paoay_root / "sentinel2/imagery_stage.json"
dynamic_world_root = paoay_root / "dynamic_world"
dynamic_world_pointer = dynamic_world_root / "dynamic_world_stage.json"
write_stage_pointer(dynamic_world_pointer, status="pending", reason="Notebook 09 has not finished the current Dynamic World request")
stages = ["Read matching imagery plan", "Check local Dynamic World", "Prepare probability inputs", "Verify Dynamic World handoff", "Summary and next action"]
notebook_progress = pipeline_progress.start_notebook(
    "09_paoay_dynamic_world", stages, report_folder.parent / "progress", {"mode": dynamic_world_mode},
)


## 2. Read matching dates and grid

Dates and extent come from notebook 08’s verified handoff. To change the period, run notebook 08 first. A missing or pending imagery handoff stops this notebook.


In [ ]:
pipeline_progress.begin_stage("Read matching imagery plan")
imagery_handoff = resolve_stage_pointer(imagery_pointer)
request = dynamic_world.prepare_request(imagery_handoff, dynamic_world_root)
print("Period:", request["period"]["start"], "to", request["period"]["end"], "(end exclusive)")
print("Grid CRS:", request["settings"]["grid"]["crs"])
print("Pixel size: 10 m; half-month inputs: 8")
print("Request folder:", request["folder"])
show_table(request["period"]["intervals"], title="Inherited half-month periods")
pipeline_progress.end_stage()


## 3. Check local inputs

Only files matching this imagery handoff and processing recipe are reused. Changed or unpaired saved files require inspection and are never automatically replaced. Sentinel-2 imagery remains untouched.


In [ ]:
pipeline_progress.begin_stage("Check local Dynamic World")
local_status = dynamic_world.cache_status(request)
show_table(local_status, title="Matching local Dynamic World inputs")
missing_composites = sum(row["status"] == "missing" for row in local_status)
print("Missing composites:", missing_composites)
pipeline_progress.end_stage()


## 4. Prepare probability inputs

Preserve the provider’s cloud/shadow masks and require all nine probability bands. Align observations to the saved grid with nearest-neighbour sampling, mosaic same-date tiles, then average daily probabilities within each half-month. This keeps probabilities together and counts each date once.

Each TIFF has nine probabilities plus `observation_count`. Probabilities stay in **0–1** with no reflectance scaling. Empty bins retain nodata and zero counts. Labels are not averaged, and no confidence threshold is selected here.


In [ ]:
pipeline_progress.begin_stage("Prepare probability inputs")
earth_engine = None
if dynamic_world_mode == "download_missing" and missing_composites:
    project = local_settings.get("earth_engine_project", "").strip()
    if not project or project == "YOUR_EARTH_ENGINE_PROJECT":
        raise ValueError("Set your Earth Engine project in settings.local.json before downloading")
    from crop_pipeline.notebook_setup import configure_certificates
    configure_certificates()
    import ee
    try:
        ee.Initialize(project=project)
    except ee.EEException:
        ee.Authenticate()
        ee.Initialize(project=project)
    earth_engine = ee

def composite_progress(items):
    return pipeline_progress.track(items, "Prepare Dynamic World inputs", unit="composite")

dynamic_world_handoff = dynamic_world.collect(
    request, mode=dynamic_world_mode, earth_engine=earth_engine, iterator=composite_progress,
)
pipeline_progress.end_stage()


## 5. Verify and save the handoff

Check all eight raster/receipt pairs, grid alignment, probability ranges/sums, daily counts and the Sentinel-2 dependency. Coverage describes available predictions, not crop accuracy. No missing period is filled using another date.


In [ ]:
pipeline_progress.begin_stage("Verify Dynamic World handoff")
world_bundle = dynamic_world.load_dynamic_world_bundle(dynamic_world_handoff)
coverage_rows = []
for interval in request["period"]["intervals"]:
    record = dynamic_world.cached_interval(request, interval)
    coverage_rows.append({"interval": interval["name"], "available_dates": len(record["available_dates"]), **record["quality"]})
show_table(coverage_rows, title="Dynamic World coverage inside the study boundary")
summary = world_bundle["manifest"]["summary"]
print("ROI pixels:", summary["roi_pixels"])
print("Pixels observed in at least one bin:", summary["any_bin_observed_pixels"])
print("Pixels observed in all eight bins:", summary["all_eight_bins_observed_pixels"])
print("Empty half-month bins:", summary["empty_bins"])
write_stage_pointer(dynamic_world_pointer, dynamic_world_handoff)
pipeline_progress.end_stage()


## 6. Summary and next action

Class probabilities remain available for water, trees, grass, flooded vegetation, crops, shrub/scrub, built, bare and snow/ice. Bare does not establish fallow; crops does not identify carrot, cabbage or potato.


In [ ]:
pipeline_progress.begin_stage("Summary and next action")
print("Verified Dynamic World handoff:", dynamic_world_handoff)
print("Current Dynamic World pointer:", dynamic_world_pointer)
print("Prediction inputs only: no land-cover decisions or accuracy claims.")
print("Next: notebook 10 will summarize land-cover guidance in a separate batch.")
print("Dynamic World attribution:", dynamic_world.ATTRIBUTION)
print("Detail log:", notebook_progress.output.path)
pipeline_progress.end_stage()
pipeline_progress.finish_notebook()
